# Step 7 Screen Fusion

This notebook fuses the two useful Step 6 screen models.
The search stays simple: weighted probability averaging plus threshold tuning on OOF predictions.


## Environment Check

The setup cell also gives us a small helper for finding the latest matching campaign.


In [3]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

EXPECTED_ENV = 'usc-nlp'
if EXPECTED_ENV not in sys.executable.lower():
    raise RuntimeError(f'This notebook should run inside the {EXPECTED_ENV} conda environment. Current executable: {sys.executable}')


def detect_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'group17_runtime.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the notebook location.')


def find_latest_run(root: Path, prefix: str) -> Path:
    candidates = [path for path in root.iterdir() if path.is_dir() and path.name.startswith(prefix)]
    if not candidates:
        raise FileNotFoundError(f'No run found in {root} with prefix {prefix!r}.')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def run_command(cmd: list[str]) -> None:
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


PROJECT_DIR = detect_project_dir(Path.cwd().resolve())
print(f'Project directory: {PROJECT_DIR}')
print(f'Python executable: {sys.executable}')


Project directory: D:\CS544-Group17-Project
Python executable: f:\anaconda\envs\usc-nlp\python.exe


## Source Campaign

Step 7 reads the newest Step 6 submission run so the chain stays explicit.


In [4]:
STEP6_RUN_ID = 'exp_s61_backbone_refresh_submission'
STEP7_SCRIPT = PROJECT_DIR / 'group17_step7_screen_fusion.py'
STEP6_ROOT = PROJECT_DIR / 'step6_outputs'
OUTPUT_ROOT = PROJECT_DIR / 'step7_outputs'
RUN_ID = 'exp_s71_screen_fusion_submission'
FUSION_ID = 's71_step6_screen_blend_v1'

STEP6_RUN_DIR = find_latest_run(STEP6_ROOT, STEP6_RUN_ID)
print(f'Source Step 6 campaign: {STEP6_RUN_DIR}')

display(pd.read_csv(STEP6_RUN_DIR / 'campaign_summary.csv'))


Source Step 6 campaign: D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211


,experiment_id,stage,model_name,input_name,cv_F1,cv_F1_std,cv_Precision,cv_Recall,cv_Accuracy,tuned_F1,best_threshold,output_dir,status
0,s61_bertweet_kwpair_anchor_v1,screen,vinai/bertweet-base,text + keyword pair,0.810673,0.017796,0.844121,0.781274,0.845128,0.813303,0.44,D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211\s61_bertweet_kwpair...,completed
1,s62_twitter_roberta_kwpair_v1,screen,cardiffnlp/twitter-roberta-base,text + keyword pair,0.806492,0.012941,0.832948,0.782841,0.840331,0.807617,0.59,D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211\s62_twitter_roberta...,completed


## Launch Step 7

The command points to the Step 6 campaign directly, so it does not depend on unrelated runs in the repo.


In [5]:
step7_cmd = [
    sys.executable,
    str(STEP7_SCRIPT),
    '--campaign-dir', str(STEP6_RUN_DIR),
    '--run-id', RUN_ID,
    '--fusion-id', FUSION_ID,
]
run_command(step7_cmd)
STEP7_RUN_DIR = find_latest_run(OUTPUT_ROOT, RUN_ID)
print(f'Step 7 run directory: {STEP7_RUN_DIR}')


Running command:
f:\anaconda\envs\usc-nlp\python.exe D:\CS544-Group17-Project\group17_step7_screen_fusion.py --campaign-dir D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211 --run-id exp_s71_screen_fusion_submission --fusion-id s71_step6_screen_blend_v1
Step 7 run directory: D:\CS544-Group17-Project\step7_outputs\exp_s71_screen_fusion_submission_local_20260424_083551


## Weight Search

The weight table shows which blend worked best on the OOF sweep.


In [6]:
weight_path = STEP7_RUN_DIR / FUSION_ID / f'{FUSION_ID}_weight_search.csv'
weight_search = pd.read_csv(weight_path)
display(weight_search.head(10))


,primary_id,secondary_id,primary_weight,secondary_weight,default_F1,best_threshold,tuned_F1,tuned_Precision,tuned_Recall,tuned_Accuracy
0,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.10,0.90,0.806706,0.57,0.808253,0.846365,0.773425,0.843929
1,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.11,0.89,0.806899,0.61,0.808707,0.853463,0.768411,0.845395
2,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.12,0.88,0.807029,0.61,0.808777,0.854007,0.768098,0.845528
3,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.13,0.87,0.807029,0.51,0.808662,0.834835,0.784080,0.842196
4,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.14,0.86,0.807612,0.55,0.808747,0.843718,0.776559,0.843796
5,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.15,0.85,0.807742,0.52,0.808938,0.836851,0.782827,0.842730
6,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.16,0.84,0.807872,0.55,0.809205,0.844346,0.776872,0.844196
7,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.17,0.83,0.808518,0.49,0.809593,0.832230,0.788154,0.842330
8,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.18,0.82,0.808971,0.53,0.809810,0.840526,0.781260,0.843929
9,s61_bertweet_kwpair_anchor_v1,s62_twitter_roberta_kwpair_v1,0.19,0.81,0.809808,0.50,0.809808,0.834441,0.786587,0.842863


## Final Step 7 Metrics

This summary captures the fused model that moves forward into Step 9.


In [7]:
summary_path = STEP7_RUN_DIR / FUSION_ID / f'{FUSION_ID}_summary.json'
step7_summary = read_json(summary_path)
final_step7 = pd.DataFrame([step7_summary['tuned_threshold']])
display(final_step7)
print(f"Primary weight: {step7_summary['primary_weight']}")
print(f"Secondary weight: {step7_summary['secondary_weight']}")
print(f"Best threshold: {step7_summary['best_threshold']}")


,F1,Precision,Recall,Accuracy
0,0.815969,0.84618,0.787841,0.84886


Primary weight: 0.69
Secondary weight: 0.31
Best threshold: 0.5
